In [7]:
import time
import json
import xgboost as xgb
import torch
import numpy as np
import joblib
from pathlib import Path
from credit_risk.features import load_features
from credit_risk.modeling.mlp import MLP

In [2]:
cwd = Path.cwd()
project_root = cwd.parent
nn_path = project_root / 'models' / 'mlp'
xgb_path = project_root / 'models' / 'tuned_xgb'

In [3]:
feature_splits = load_features()

2026-06-29 21:13:39.885 | INFO     | credit_risk.features:load_features:263 - Loading the processed features...
2026-06-29 21:13:40.265 | INFO     | credit_risk.features:load_features:270 - Loaded successfully!


In [4]:
X_train = feature_splits['train'][0].to_numpy()
y_train = feature_splits['train'][1].to_numpy().ravel()
X_val = feature_splits['val'][0].to_numpy()
y_val = feature_splits['val'][1].to_numpy().ravel()
X_test = feature_splits['test'][0].to_numpy()
y_test = feature_splits['test'][1].to_numpy().ravel()

In [5]:
with open(nn_path / 'architecture.json') as f:
    archi = json.load(f)

In [6]:
# loading the models
xgb_model = joblib.load(xgb_path / 'model.pkl')
state_dict = torch.load(nn_path / 'model_state.pt')
mlp_model = MLP(input_dim=archi['input_dim'], hidden_dim=archi['hidden_dim'], dropout=archi['dropout'])
mlp_model.load_state_dict(state_dict=state_dict)

<All keys matched successfully>

In [12]:
X_test[:1].shape

(1, 147)

In [26]:
# latency check
# xgb

# warm up
for _ in range(10):
    _ = xgb_model[1].predict_proba(X_test[:1])
    
n_trials = 100
times = []
for _ in range(n_trials):
    start = time.perf_counter()
    _ = xgb_model[1].predict_proba(X_test[:1])
    end = time.perf_counter()
    times.append((end - start) * 1000)
    
print(f"median: {np.median(times):.3f} ms")
print(f'P95: {np.percentile(times, 95):.3f} ms')

median: 0.279 ms
P95: 0.599 ms


In [27]:
# MLP

single_tensor = torch.from_numpy(X_test[:1].astype(np.float32))

mlp_model.eval()
with torch.no_grad():
    # warm up
    for _ in range(10):
        _ = torch.sigmoid(mlp_model(single_tensor))
        
    n_trials = 100
    times = []
    for _ in range(n_trials):
        start = time.perf_counter()
        _ = torch.sigmoid(mlp_model(single_tensor))
        end = time.perf_counter()
        times.append((end - start) * 1000)
        
print(f"median: {np.median(times):.3f} ms")
print(f'P95: {np.percentile(times, 95):.3f} ms')

median: 0.029 ms
P95: 0.032 ms


In [29]:
# batch latency
# xgb

# warm up
for _ in range(3):
    _ = xgb_model[1].predict_proba(X_test[:1000])
    
n_trials = 50
times = []
for _ in range(n_trials):
    start = time.perf_counter()
    _ = xgb_model[1].predict_proba(X_test[:1000])
    end = time.perf_counter()
    
    times.append((end - start) * 1000)
    
print(f"Median: {np.median(times):.3f} ms")
print(f"per-row: {(np.median(times)/1000):.3f} ms")

Median: 1.583 ms
per-row: 0.002 ms


In [ ]:
# mlp

single_batch = torch.from_numpy(X_test[:1000].astype(np.float32))

with torch.no_grad():
    # warmup
    for _ in range(3):
        _ = torch.sigmoid(mlp_model(single_batch))
        
    n_trials = 50
    times = []
    for _ in range(50):
        start = time.perf_counter()
        _ = torch.sigmoid(mlp_model(single_batch))
        end = time.perf_counter()
        
        times.append((end - start) * 1000)
        
print(f"Median: {np.median(times):.3f} ms")
print(f"per-row: {(np.median(times)/1000):.3f} ms")